In [8]:
!pip install pyspark

In [12]:
import os
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, when, from_unixtime

# 1. DỌN DẸP SESSION CŨ
try:
    spark.stop()
except:
    pass

# 2. KHAI BÁO THÔNG TIN R2
ACCESS_KEY = "352e66be1b3cb7fad1866c153222faf2"
SECRET_KEY = "6b97ebc8310262f21fcb5e1d315ce7b3224f73a39f7813ebb38c21fa4ab30ea2"
ACCOUNT_ID = "02422b2aa3d6c7acdf824dfd9259afb2"

R2_ENDPOINT = f"https://{ACCOUNT_ID}.r2.cloudflarestorage.com"
BUCKET_NAME = "amazon-reviews-test"
CATEGORY = "Tools_and_Home_Improvement"

source_path = f"s3a://{BUCKET_NAME}/parquet-data/review/{CATEGORY}/"
dest_path = f"s3a://{BUCKET_NAME}/data-preprocess/review/{CATEGORY}/"

# 3. KHỞI TẠO SPARK (CHẶN TOÀN BỘ LỖI PARSE THỜI GIAN)
print("⏳ Đang khởi tạo Spark Session...")
spark = SparkSession.builder \
    .appName("Preprocess_Amazon_Reviews") \
    .config("spark.jars.packages", "org.apache.hadoop:hadoop-aws:3.3.4,com.amazonaws:aws-java-sdk-bundle:1.12.262") \
    .config("spark.hadoop.fs.s3a.access.key", ACCESS_KEY) \
    .config("spark.hadoop.fs.s3a.secret.key", SECRET_KEY) \
    .config("spark.hadoop.fs.s3a.endpoint", R2_ENDPOINT) \
    .config("spark.hadoop.fs.s3a.path.style.access", "true") \
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "true") \
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") \
    .config("spark.hadoop.fs.s3a.aws.credentials.provider", "org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider") \
    .config("spark.hadoop.fs.s3a.connection.timeout", "60000") \
    .config("spark.hadoop.fs.s3a.connection.establish.timeout", "5000") \
    .config("spark.hadoop.fs.s3a.connection.request.timeout", "30000") \
    .config("spark.hadoop.fs.s3a.threads.keepalivetime", "60") \
    .config("spark.hadoop.fs.s3a.multipart.purge", "false") \
    .config("spark.hadoop.fs.s3a.multipart.purge.age", "86400") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")

# 4. ĐỌC DỮ LIỆU
print(f"📥 Đang đọc toàn bộ file Parquet từ: {source_path}")
df_raw = spark.read.parquet(source_path)

# 5. TIỀN XỬ LÝ DỮ LIỆU
print("⚙️ Đang tiến hành tiền xử lý (Ép kiểu & Làm sạch)...")
df_cleaned = df_raw \
    .withColumn("rating", col("rating").cast("float")) \
    .withColumn("helpful_vote", col("helpful_vote").cast("integer")) \
    .withColumn("verified_purchase", 
                when(col("verified_purchase").isin("TRUE", "true", "True"), True)
                .otherwise(False)) \
    .withColumn("timestamp", from_unixtime(col("timestamp").cast("double") / 1000).cast("timestamp")) \
    .fillna({"text": "", "title": ""}) \
    .dropna(subset=["user_id", "parent_asin", "rating"])

# 6. GHI DỮ LIỆU LÊN R2
print(f"📤 Đang ghi dữ liệu đã xử lý về: {dest_path}")
df_cleaned.write.mode("overwrite").parquet(dest_path)

print("✅ QUÁ TRÌNH TIỀN XỬ LÝ HOÀN TẤT!")

⏳ Đang khởi tạo Spark Session...
📥 Đang đọc toàn bộ file Parquet từ: s3a://amazon-reviews-test/parquet-data/review/Tools_and_Home_Improvement/


⚙️ Đang tiến hành tiền xử lý (Ép kiểu & Làm sạch)...
📤 Đang ghi dữ liệu đã xử lý về: s3a://amazon-reviews-test/data-preprocess/review/Tools_and_Home_Improvement/


✅ QUÁ TRÌNH TIỀN XỬ LÝ HOÀN TẤT!
